In [ ]:
# === Setup ===
# Runtime: <3m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Reference solution — Text Classification

Split → TF-IDF fit on train only → Logistic Regression → Macro F1 → submission.

In [ ]:
texts=np.array(["phim rất hay","trải nghiệm tuyệt vời","dịch vụ tốt","mình rất thích","quá tệ","dịch vụ kém","không bao giờ quay lại","thất vọng"]*8)
labels=np.array([1,1,1,1,0,0,0,0]*8); rng=np.random.default_rng(42); order=rng.permutation(len(texts)); texts,labels=texts[order],labels[order]
cut=48; train_text,val_text=texts[:cut],texts[cut:]; y_train,y_val=labels[:cut],labels[cut:]
assert set(y_train)=={0,1}; print("train/val",len(train_text),len(val_text))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
vectorizer=TfidfVectorizer(ngram_range=(1,2)); Xtr=vectorizer.fit_transform(train_text); Xva=vectorizer.transform(val_text)
model=LogisticRegression(random_state=42,max_iter=300).fit(Xtr,y_train); pred=model.predict(Xva); score=f1_score(y_val,pred,average="macro")
submission=np.c_[np.arange(len(pred)),pred]; assert score>.9 and submission.shape==(len(y_val),2)
print("macro_f1",score,"vocab",len(vectorizer.vocabulary_))